In [1]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate_hkqai_done").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict_ccpvdz"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero
        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = data_scf[col[0]] - data_cc[col[0]]
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["ai"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["ai"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

    wtmad_1_subset.loc["summary", (data_path_name, "Processed")] = "--"
    wtmad_2_subset.loc["summary", (data_path_name, "Processed")] = "--"
    for d3_name in ["", "_d3bj", "_d3zero"]:
        wtmad_1_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_1_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        for name_set in full_subset_dict.keys():
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]

print("Summary")
display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# save summary to excel with date
df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 12.609903921023943 kcal/mol, 116 in 1424849_W4_11
  -1 * W4_11-p4  -1 * -13.835936387768015  4 * W4_11-p  4 * -0.3065081166860182
Top 2 AI: 5.341987040214008 kcal/mol, 136 in 1424849_W4_11
  -1 * W4_11-foof  -1 * -4.328057378443191  2 * W4_11-f  2 * 0.6256503599724965  2 * W4_11-o  2 * -0.11868552908708807
Top 3 AI: 5.219463497924153 kcal/mol, 46 in 1424849_W4_11
  -1 * W4_11-alf3  -1 * -3.2680325650726445  1 * W4_11-al  1 * 0.07447985294857062  3 * W4_11-f  3 * 0.6256503599724965
Top 4 AI: 4.821493316645501 kcal/mol, 135 in 1424849_W4_11
  -1 * W4_11-cloo  -1 * -4.751233361486811  1 * W4_11-cl  1 * 0.30763101333286613  2 * W4_11-o  2 * -0.11868552908708807
Top 5 AI: 4.2865408250218024 kcal/mol, 63 in 1424849_W4_11
  -1 * W4_11-sif  -1 * -3.7919159706216305  1 * W4_11-si  1 * -0.13102550557232462  1 * W4_11-f  1 * 0.6256503599724965
Top 1 DFT: 64.9391765203327 kcal/mol
Top 2 DFT: 64.82811637483246 kcal/mol
Top 3 DFT: 58.7866408124537 kcal/mol
Top 4 DFT: 55.61089226667

data_path       1424849                                            \
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip   
summary         0.32847      0.352175      0.031172       0.03176   
W4_11          0.128193      0.146661      0.023594      0.021439   
G21EA          0.079752      0.090296      0.012511      0.012762   
G21IP          0.081996      0.086805      0.005995      0.006154   
DIPCS10        0.101593      0.107438      0.005017      0.003622   
PA26           0.268845        0.2982      0.034744      0.038429   
SIE4x4         0.071736      0.093996      0.003569      0.001311   
ALKBDE10       0.098409       0.10564      0.065934      0.061711   
YBDE18         0.257196      0.276239      0.041119      0.041601   
AL2X6          0.395256      0.396501      0.002764      0.002336   
HEAVYSB11      0.288568      0.278481      0.011631      0.009043   
NBPRC          0.237834      0.252724      0.032958      0.032482   
ALK8           0.202733      0.187227      0.040614      0.040223   
RC21           0.244294      0.258984      0.071539      0.082447   
G2RC           0.163856      0.181463      0.015502      0.015617   
BH76RC          0.12608       0.14036      0.055976      0.057338   
FH51           0.342554      0.346443      0.024006      0.026834   
TAUT15         0.369822      0.413544      0.067899      0.069453   
DC13           0.390599      0.397856      0.020702      0.019322   
MB16_43        0.500446      0.536587      0.084707      0.086338   
DARC           0.426179      0.443789      0.014191      0.019082   
RSE43          0.209548      0.233351      0.034582      0.033645   
BSR36          0.534747      0.495778      0.004311      0.001683   
CDIE20         0.370491       0.36033      0.039229      0.043548   
ISO34           0.29061      0.298776      0.025594      0.026941   
PArel          0.522926      0.599811      0.074776      0.079046   
BH76            0.12608       0.14036      0.055976      0.057338   
BHPERI         0.329315      0.326909      0.031284      0.029987   
BHDIV10        0.342553      0.361694      0.049657      0.049067   
INV24          0.778226      0.784024      0.038296      0.027518   
BHROT27        0.289291      0.303021      0.021441      0.018507   
PX13           0.215428      0.278637      0.012669       0.01315   
WCPT18         0.197421      0.224154      0.060305      0.057861   
RG18           0.204128      0.206046        0.0023      0.002641   
ADIM6          0.480426      0.434301      0.000826       0.00024   
S22            0.385426      0.405222      0.025222      0.024365   
S66            0.331414      0.347655      0.022744      0.025355   
WATER27        0.393804      0.547345       0.02721      0.033217   
CARBHB12       0.176663      0.195377      0.033396      0.036954   
PNICO23        0.208847      0.233577        0.0258      0.023517   
HAL59          0.399949      0.412527      0.042605      0.036969   
AHB21          0.108228      0.129997      0.032458      0.031459   
CHB6           0.162305       0.16005      0.019153      0.019259   
IL16           0.290156      0.335812      0.049496       0.05021   
IDISP           0.89541      0.816401      0.002265      0.000619   
ICONF          0.464768      0.533359      0.023171      0.022986   
ACONF          0.384884      0.355398      0.003603      0.001781   
Amino20x4      0.723524      0.806256       0.02953      0.037327   
PCONF21         1.05244      1.184712      0.058859      0.077448   
MCONF          0.933391      0.987572      0.038508      0.045477   
SCONF          0.606237      0.725383      0.033587      0.024507   
BUT14DIOL      0.338723      0.380054      0.025886       0.02199   

data_path       3036943                                            \
Ele type  error_scf_ele error_dft_ele error_scf_dip error_dft_dip   
summary        0.272361      0.353225      0.032258       0.03176   
W4_11          0.161587      0.148743      0.022614      0.021439 

MAE


data_path   1424849                                                      \
Disp type        AI        DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1       2.002477  13.710857  1.935526   13.7889  1.859815  13.685393   
sub2       5.510109   6.612164  5.412513  6.359139  5.458901   6.408048   
sub3       2.617318   6.261376  2.816465  6.625846  2.938387   6.678362   
sub4        1.93524   3.272197  1.474342  3.440323  1.710948   3.742517   
sub5       1.626727   1.321288  1.369759  0.815469  1.357554   0.809664   

data_path             3036943                       ...    206726             \
Disp type Processed        AI        DFT   AI_D3BJ  ... AI_D3ZERO DFT_D3ZERO   
sub1           DONE  5.772845  13.710857  5.904607  ...  6.307889  13.713797   
sub2           DONE  7.919127   6.612128     6.749  ...  5.817709   6.757695   
sub3           DONE  4.636265   6.261376  5.246788  ...  4.328635   6.934991   
sub4           DONE   3.14773   3.272197  4.543588  ...  3.362683   4.988172   
sub5           DONE  1.325602   1.321288   0.77258  ...  1.102017   0.875423   

data_path             1428227                                            \
Disp type Processed        AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO   
sub1           DONE  6.209586  13.710857  5.942741  14.259701  5.870916   
sub2           DONE  8.731272   6.612128  6.479642   9.130035  5.850646   
sub3           DONE  3.494263   6.261376  3.543113   7.355579  3.339518   
sub4           DONE  2.090411   3.272197  2.775221    5.03074  2.773315   
sub5          7 / 8  1.564185   1.321288  0.930474   0.928323  0.936663   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1       13.713797      DONE  
sub2        6.757695      DONE  
sub3        6.934991      DONE  
sub4        4.988172      DONE  
sub5        0.872793      DONE  

[5 rows x 35 columns]

wtmad_1


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        3.239113   7.420922   2.966428   7.224551   2.815004   7.166359   
sub2       13.142366   12.53052  12.443324  11.377537  12.073631  10.891896   
sub3        4.832263   6.984991   5.052323    7.34316   5.207324    7.32152   
sub4       11.556831  11.029773   7.478943   7.022789   7.675787   7.720437   
sub5       15.335367  11.571826  12.722907   6.873743  12.540889   6.757453   
summary    48.105939  49.538032  40.663925   39.84178  40.312635  39.857666   

data_path              3036943                        ...     206726  \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE   6.543701   7.420922   5.841371  ...   7.142963   
sub2           DONE  11.307039  12.530489   8.468475  ...   9.842185   
sub3           DONE   5.546066   6.984991   6.136313  ...   5.880589   
sub4           DONE  10.448711  11.029773  12.102006  ...   9.275534   
sub5           DONE   11.77592  11.571826   6.644581  ...   9.973326   
summary          --  45.621438  49.538002  39.192745  ...  42.114597   

data_path                         1428227                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        7.102094      DONE   7.490622   7.420922   6.773322   7.414578   
sub2       10.005264      DONE  12.569598  12.530489  10.018883   9.968131   
sub3        7.521852      DONE   4.827681   6.984991    4.88772   8.150912   
sub4       13.798614      DONE  11.925636  11.029773    8.32846   14.87552   
sub5         7.61332     7 / 8  14.263235  11.571826   7.968527   8.092645   
summary    46.041146        --  51.076772  49.538002  37.976913  48.501786   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        6.744158   7.102094      DONE  
sub2       10.339683  10.005264      DONE  
sub3        4.526504   7.521852      DONE  
sub4        7.404226  13.798614      DONE  
sub5        7.985224   7.523212      DONE  
summary    36.999795  45.951037        --  

[6 rows x 35 columns]

wtmad_2


data_path    1424849                                                        \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1        1.438234   4.040696   1.347877   4.009887  1.298132   3.993215   
sub2        3.339073   3.635331    3.00695   3.050633  2.887856   2.955369   
sub3        1.303233   2.611064   1.379492   2.754823  1.432794   2.775157   
sub4        4.683102   4.159493   3.352776   3.139343  3.830024   3.673649   
sub5        6.331811    4.66236   5.509119   2.969375  5.489474   2.874701   
summary    17.095452  19.108945  14.596213  15.924061  14.93828   16.27209   

data_path              3036943                        ...     206726  \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1           DONE   2.856326   4.040696   2.698245  ...   3.319474   
sub2           DONE   3.372183   3.635301   1.993832  ...   2.238128   
sub3           DONE   2.042813   2.611064   2.287867  ...   1.734701   
sub4           DONE   4.114693   4.159493   5.957429  ...   4.681643   
sub5           DONE   4.687978    4.66236   2.783135  ...   3.927966   
summary          --  17.073993  19.108915  15.720508  ...  15.901913   

data_path                         1428227                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        4.001193      DONE   3.343814   4.040696   3.112167   4.111446   
sub2        2.552948      DONE   3.692046   3.635301    2.35309   2.376009   
sub3        2.881875      DONE   1.455183   2.611064   1.470681   3.040013   
sub4        6.639136      DONE   4.529564   4.159493   4.480017   6.856534   
sub5        3.167984     7 / 8   5.725369    4.66236   3.557303   3.511844   
summary    19.243137        --  18.745976  19.108915  14.973258  19.895846   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        3.080427   3.989769      DONE  
sub2          2.6181   2.545658      DONE  
sub3        1.392062   2.873647      DONE  
sub4         4.28232    6.62018      DONE  
sub5        3.413984    3.19725      DONE  
summary    14.786894  19.226505        --  

[6 rows x 35 columns]

Summary of Subset
MAE


data_path    1424849                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       1.028249  29.506863   1.223261  29.989789   1.151098  29.747048   
G21EA       0.816732   9.754522   0.816599    9.75468   0.820867   9.756932   
G21IP       0.694714   8.953334    0.69616   8.951545    0.68818   8.952599   
DIPCS10     2.042735  12.310852   2.039936   12.31518   2.045245  12.283687   
PA26        3.009087   2.200766   2.985201   2.058032   2.957953   2.055317   
SIE4x4      3.430805  21.908516   3.542007  22.097166   3.562974  22.161897   
ALKBDE10    0.959132  18.125246   0.989582  18.277559   0.952777  18.132767   
YBDE18      5.939045   8.145393   5.216198   7.828353   4.841699   7.695204   
AL2X6       6.140013   5.659145   4.464174   3.449937   3.988804   3.300348   
HEAVYSB11   2.024005   5.391476   1.791244   5.463331   1.721678   5.379671   
NBPRC       3.058861    2.23255   2.703933   1.870626   2.562871   2.028757   
ALK8        4.403777   4.400063   3.334795   3.350254   2.788735   2.953216   
RC21        2.456062   4.820926   1.923738    5.39421    1.82928   5.525602   
G2RC        1.887093   5.917203   1.969408   6.249306   1.995017   6.227185   
BH76RC       0.85212   3.484561   0.891385    3.50012   0.922322   3.516546   
FH51        2.464205   3.703657   2.169993   3.443826   2.129573   3.347368   
TAUT15      1.702465    2.16453    1.66463   2.155289   1.600571   2.170609   
DC13        6.170057  13.090861   6.241887  12.478213   5.863507  11.959009   
MB16_43     13.39941  15.461604  15.297752  18.175348  16.246698  18.708523   
DARC        7.823359   10.75415   5.637199   7.786582   4.993859   7.610198   
RSE43        3.11738     3.1576   3.042746   3.052617   2.897426   2.895709   
BSR36        4.98276    8.40118   3.199812   5.170108   2.849067   5.230489   
CDIE20      1.450234   1.599507   1.451078   1.511089   1.479492   1.421578   
ISO34       2.020035   2.001743   1.914195   1.837375   1.829508   1.772453   
PArel       3.015433   1.743933   2.988405    1.73941   2.944914    1.65864   
BH76        2.803583   9.107946   2.865363   9.384703   2.949706   9.471921   
BHPERI      2.657111   3.268992   3.650976    4.91131   4.133776   5.201979   
BHDIV10     4.741552   6.277706   4.837814   6.576988   4.819558   6.463527   
INV24       3.393792   2.697333    3.50674   2.430052   3.504461   2.367796   
BHROT27     1.713103   0.853379   1.717399   0.843049   1.745492   0.815488   
PX13        1.059156  11.042023   1.034643  11.281052   1.127794  11.125104   
WCPT18      2.039614   7.967151    2.29673    8.38753   2.461054   8.465437   
RG18         0.30936   0.230018   0.307671   0.272271   0.438413   0.363283   
ADIM6       3.598266   3.059029   1.159053   0.069974   1.006802   0.047161   
S22         2.728845   2.291252   2.115574   1.027824   2.108266   1.185279   
S66         2.326488   1.894709    1.01562   0.729648   0.940707   0.863238   
WATER27      3.02651  16.299413   2.824412  20.709459   4.382597  22.096474   
CARBHB12    0.484944   1.632408   0.473346   2.236748   0.556368   2.356687   
PNICO23     0.840943   0.776429   0.940904   1.237735    0.98267   1.220119   
HAL59       2.035403   1.649786   1.813837   1.431131   1.991528   1.687656   
AHB21       1.666768   2.453821   1.835619   2.836357   1.986367   2.982124   
CHB6        1.766454    1.80532   1.689088   2.042249   1.499007   2.186047   
IL16        1.467889   1.021067   2.011731   2.478095   2.624917    3.12773   
IDISP      13.356085  13.125475   9.748251   6.829682   9.297146   6.724485   
ICONF       0.946016   0.413804   0.923008   0.398194   0.896902   0.460113   
ACONF       3.016487   0.546621   2.780804   0.134491    2.73024   0.112191   
Amino20x4   1.243276   0.656235   1.214437   0.533098   1.214533   0.545417   
PCONF21      2.09681   3.214782   1.225373   1.470386   1.250642   1.109308   
MCONF       1.596087    1.95602   1.061923   0.627